# Palma de Mallorca: baseline results

February 2022 weekday, car only, LTM traffic model, day-to-day assignment (gap rule, 9 iterations).
Baseline run `palma_feb2022_baseline_iter8` (workspace HANDOFF UPDATE 72-74):
- network 07 (HCM capacities, lanes per direction);
- demand v4 (building-volume trip ends + exchange trips with the rest of Mallorca);
- the 3 LTM node rules.

Two earlier chains are shown for comparison, with the same network, node rules and protocol:

| Chain | Demand |
|---|---|
| `gap_demand_v2_node_rules` | zone fix only |
| `gap_demand_v3_node_rules` | + day-mean fix and building weights, no exchange |

Runs from a clean kernel in the `nomad` conda env. Figures go to `reports/2026-11-25_palma_baseline_results/figures/` in the workspace.

In [ ]:
import os, sys, json, csv, statistics
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm, Normalize

REPO = '/home/fbellisardi/code/nomad-walk'
W = '/data/workspaces/fbellisardi/nomad'
sys.path.insert(0, f'{REPO}/python')
from nomad_tools import io, chain_summary, count_compare

RUNS = f'{W}/runs'
FIG = f'{W}/reports/2026-11-25_palma_baseline_results/figures'
CACHE = f'{W}/reports/2026-11-25_palma_baseline_results/cache'
os.makedirs(FIG, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
SECTIONS = f'{W}/reports/2026-11-12_day_to_day_assignment/imd/sections.csv'
GRAPH = f'{W}/inputs/palma_de_mallorca/palma_network_07_hcm_capacity.bin'
OD = f'{W}/inputs/palma_de_mallorca/palma_demand_feb2022_weekday_v4_exchange_unclipped.csv'  # the OD the baseline ran on
P = 'palma_feb2022_'
# chain -> (label, colour, marker). Colour follows the chain in every figure; the baseline is the dark ink slot.
CHAINS = {'gap_demand_v2_node_rules': ('demand v2 (zone fix)', '#0072b2', 's'),
          'gap_demand_v3_node_rules': ('demand v3 (weights)', '#c27c00', '^'),
          'baseline': ('baseline (v4: + exchange)', '#1b1b1b', 'o')}
run = lambda c, k=8: f'{RUNS}/{P}{c}_iter{k}'

## Style

In [ ]:
INK, INK2, GRID = '#1b1b1b', '#5a5a5a', '#d9d9d9'
mpl.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 7.5, 'axes.titlesize': 8, 'axes.labelsize': 7.5, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
    'grid.linewidth': 0.4, 'lines.linewidth': 1.3, 'lines.markersize': 4, 'legend.frameon': False,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'figure.dpi': 110, 'savefig.dpi': 300})
SEQ = 'cividis'   # sequential maps: colour-vision safe, monotone lightness

def save(fig, name):
    for ext in ('pdf', 'svg', 'png'):
        fig.savefig(f'{FIG}/{name}.{ext}', bbox_inches='tight')

## 1. Load and check the headline numbers
The assertions fail if the inputs differ from the values recorded in HANDOFF UPDATE 72.

In [ ]:
logs = {c: [chain_summary.parse_log(f'{W}/logs/{P}{c}_iter{k}.o') for k in range(9)] for c in CHAINS}
sections = list(csv.DictReader(open(SECTIONS)))
imd = {c: count_compare.compare(sections, [run(c)], cache_dir=CACHE) for c in CHAINS}
ratio = {c: np.array([f[0] / float(s['imd']) for s, _, f in imd[c]]) for c in CHAINS}
od = pd.read_csv(OD)
n_base_rows = 531569                     # internal (v3) rows come first, exchange rows after (UPDATE 71)

b8 = logs['baseline'][8]
assert od.loc[od['mode'] == 'car', 'count'].sum() == 1_016_284
assert od['count'].iloc[n_base_rows:].sum() == 116_146
assert (b8['OnLink'], b8['Waiting']) == (2535, 70) and abs(b8['rg'] - 0.0175) < 1e-9
assert round(statistics.median(np.round(ratio['baseline'], 2)), 2) == 0.79
assert round(statistics.median(np.round(ratio['gap_demand_v3_node_rules'], 2)), 2) == 0.71
print('headline numbers verified:',
      {c: round(float(np.median(ratio[c])), 3) for c in CHAINS})

## 2. Convergence of the day-to-day assignment
Relative gap (RG) of each replanning step and number of agents that switched plan. iter0 is free-flow routing, so it has no RG.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6.3, 2.3), constrained_layout=True)
it = np.arange(1, 9)
for c, (lab, col, mk) in CHAINS.items():
    ax[0].plot(it, [logs[c][k]['rg'] for k in it], color=col, marker=mk, label=lab)
    ax[1].plot(it, [logs[c][k]['switched'] / 1e3 for k in it], color=col, marker=mk, label=lab)
ax[0].set(yscale='log', xlabel='iteration', ylabel='relative gap', title='a  Relative gap')
ax[1].set(yscale='log', xlabel='iteration', ylabel='agents switching plan (thousands)', title='b  Plan switches')
ax[1].legend(loc='upper right')
save(fig, 'fig1_convergence'); plt.show()

## 3. Vehicles on the network through the day
Sum of per-edge counts in every 15-min snapshot of iteration 8. It includes vehicles queued on links.

In [ ]:
def load_curve(r):
    ts = io.snapshot_times(r)
    return np.array(ts) / 3600, np.array([io.read_snapshot(r, t, None, ('count',))['count'].sum() for t in ts])
curves = {c: load_curve(run(c)) for c in CHAINS}
fig, ax = plt.subplots(figsize=(4.2, 2.4), constrained_layout=True)
for c, (lab, col, mk) in CHAINS.items():
    h, v = curves[c]
    ax.plot(h, v / 1e3, color=col, label=lab, marker=mk, markevery=8)
ax.set(xlabel='time of day (h)', ylabel='vehicles on the network (thousands)', xlim=(0, 24), xticks=range(0, 25, 3))
ax.legend(loc='upper left')
save(fig, 'fig2_network_load'); plt.show()
print({c: f'peak {curves[c][1].max():,.0f} at {curves[c][0][curves[c][1].argmax()]:.2f} h' for c in CHAINS})

## 4. Simulated flow against the IMD counts
- IMD 2022 is the **annual** average daily traffic of **all vehicles** (Consell de Mallorca). The simulation is a February weekday, cars only.
- The expected sim/IMD is therefore below 1. The 0.7-0.95 range used in the HANDOFF is an assumption, not verified.
- Shaded band: 0.8-1.2.

In [ ]:
names = [s['road'] for s in sections]
order = np.argsort(ratio['baseline'])
fig, ax = plt.subplots(1, 2, figsize=(6.6, 3.2), constrained_layout=True, gridspec_kw={'width_ratios': [1.5, 1]})
y = np.arange(len(names))
ax[0].axvspan(0.8, 1.2, color='#efefef', zorder=0, lw=0)
ax[0].axvline(1, color=INK2, lw=0.6)
for j, (c, (lab, col, mk)) in enumerate(CHAINS.items()):
    ax[0].plot(ratio[c][order], y + (j - 1) * 0.22, ls='none', marker=mk, color=col, label=lab, ms=3.5)
ax[0].set(yticks=y, yticklabels=[names[i] for i in order], xlabel='simulated / IMD (daily, both directions)',
          title='a  Ratio per count section (iteration 8)')
ax[0].grid(axis='y', visible=False); ax[0].legend(loc='lower right')
obs = np.array([float(s['imd']) for s in sections]); sim = ratio['baseline'] * obs
ax[1].plot([0, 190], [0, 190], color=INK2, lw=0.6)
ax[1].plot(obs / 1e3, sim / 1e3, 'o', color=CHAINS['baseline'][1], ms=3.5)
LBL = {'47973': (6, -2), '150361': (6, -8)}   # IMD id -> label offset (points)
for i, s_ in enumerate(sections):
    if s_['imd'] in LBL:
        ax[1].annotate(s_['road'], (obs[i] / 1e3, sim[i] / 1e3), xytext=LBL[s_['imd']], textcoords='offset points',
                       fontsize=6, color=INK2)
ax[1].set(xlabel='IMD (thousand veh/day)', ylabel='simulated (thousand cars/day)', title='b  Baseline vs IMD',
          xlim=(0, 190), ylim=(0, 190), aspect='equal')
save(fig, 'fig3_imd'); plt.show()
print('share of sections in [0.8, 1.2]:', {c: round(float(((ratio[c] >= 0.8) & (ratio[c] <= 1.2)).mean()), 2) for c in CHAINS})

## 5. Maps
Network geometry from the run's `static_network.geojson`. Panel a: daily car entries per edge. Panel b: congestion at 18:00, as the ratio of travel time to free-flow travel time on edges with vehicles.

In [ ]:
static = io.static_network(run('baseline'))
segs = [np.array(f['geometry']['coordinates']) for f in static]
eid = np.array([f['properties']['edge_id'] for f in static]); rc = np.array([f['properties']['road_class'] for f in static])
daily = io.daily_entries(run('baseline'), int(eid.max()) + 1, f'{CACHE}/daily_{P}baseline_iter8.npy')
snap = json.load(open(f"{io.snapshot_dir(run('baseline'))}/network_state_064800.geojson"))['features']
tt_ratio = np.full(int(eid.max()) + 1, np.nan)
for f in snap:
    p = f['properties']
    if p['count'] > 0 and p['free_flow_time_s'] > 0:
        tt_ratio[p['edge_id']] = p['travel_time_s'] / p['free_flow_time_s']
assert abs(daily.sum() - 90_399_913) < 1, daily.sum()     # total entries of iter 8 (eval output, UPDATE 76)

car = rc <= 14
base_lc = LineCollection([segs[i] for i in np.nonzero(car)[0]], colors='#e6e6e6', linewidths=0.2, zorder=0)
fig, ax = plt.subplots(1, 2, figsize=(7.0, 3.6), constrained_layout=True)
# a: daily flow
sel = np.nonzero(car & (daily[eid] >= 100))[0]; sel = sel[np.argsort(daily[eid[sel]])]
v = daily[eid[sel]]
lc = LineCollection([segs[i] for i in sel], array=v, cmap=SEQ + '_r', norm=LogNorm(100, v.max()),
                    linewidths=0.3 + 1.6 * (np.log10(v) - 2) / (np.log10(v.max()) - 2))
ax[0].add_collection(LineCollection(base_lc.get_segments(), colors='#e6e6e6', linewidths=0.2))
ax[0].add_collection(lc); fig.colorbar(lc, ax=ax[0], shrink=0.7, label='car entries per day (log)')
ax[0].set_title('a  Daily car flow, baseline iteration 8')
# b: congestion at 18:00
used = np.nonzero(car & np.isfinite(tt_ratio[eid]))[0]
ax[1].add_collection(LineCollection([segs[i] for i in used], colors='#9e9e9e', linewidths=0.35))   # edges in use, near free flow
sel = used[tt_ratio[eid[used]] >= 1.25]; sel = sel[np.argsort(tt_ratio[eid[sel]])]
r = np.clip(tt_ratio[eid[sel]], 1.25, 4)
lc2 = LineCollection([segs[i] for i in sel], array=r, cmap=SEQ + '_r', norm=Normalize(1.25, 4),
                     linewidths=1.6 + 0.6 * (r - 1.25))
print(f'edges in use at 18:00: {len(used):,}; with travel time >= 1.25 x free flow: {len(sel):,}')
ax[1].add_collection(LineCollection(base_lc.get_segments(), colors='#e6e6e6', linewidths=0.2))
ax[1].add_collection(lc2); fig.colorbar(lc2, ax=ax[1], shrink=0.7, label='travel time / free-flow time (1.25-4, capped)')
ax[1].set_title('b  Congestion at 18:00 (grey: in use, < 1.25 x free flow)')
for a in ax:
    a.set(xlim=(2.50, 2.85), ylim=(39.48, 39.70), aspect=1 / np.cos(np.radians(39.6)))
    a.grid(False); a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)
save(fig, 'fig4_maps'); plt.show()

Count sections on the map, coloured by baseline sim/IMD and labelled with the ratio. Same palette as the maps, centred on 1.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
fig, ax = plt.subplots(figsize=(4.6, 3.8), constrained_layout=True)
major = np.nonzero(rc <= 6)[0]
ax.add_collection(LineCollection([segs[i] for i in major], colors='#bdbdbd', linewidths=0.4))
norm = TwoSlopeNorm(vmin=0.3, vcenter=1.0, vmax=1.7); cmap = plt.get_cmap('PuOr')
NUDGE = {'49827': (-22, 2), '46004': (3, 6), '160410': (3, -9), '140172': (4, -2), '147363': (-20, 4)}  # label offsets (points)
for s, rr in zip(sections, ratio['baseline']):
    x = [float(s['lon1']), float(s['lon2'])]; yy = [float(s['lat1']), float(s['lat2'])]
    ax.plot(x, yy, color=cmap(norm(rr)), lw=2.2, solid_capstyle='round')
    off = NUDGE.get(s['imd'], (2, 2))
    ax.annotate(f'{rr:.2f}', (x[1], yy[1]), xytext=off, textcoords='offset points', fontsize=6, color=INK)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); fig.colorbar(sm, ax=ax, shrink=0.7, label='simulated / IMD')
ax.set(xlim=(2.55, 2.80), ylim=(39.52, 39.66), aspect=1 / np.cos(np.radians(39.6)), title='IMD count sections, baseline iteration 8')
ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_visible(False)
save(fig, 'fig5_imd_map'); plt.show()

Very few edges are congested at 18:00. The network carries the baseline demand almost at free flow, apart from a few junctions. This matches the low vehicles-on-network curve, and it is a possible reason for the IMD under-counts on the ring: demand is missing, not capacity. That reading is not tested here.

## 6. Demand: departure profile
Cars per hour by departure window in the baseline OD. Internal trips are the v3 rows; exchange trips are the rows added in v4. Inbound exchange windows are shifted by the outside leg.

In [ ]:
h = (od['depart_mean_s'] // 3600).clip(upper=23).astype(int)
internal = od['count'].iloc[:n_base_rows].groupby(h.iloc[:n_base_rows]).sum().reindex(range(24), fill_value=0)
exchange = od['count'].iloc[n_base_rows:].groupby(h.iloc[n_base_rows:]).sum().reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(4.2, 2.3), constrained_layout=True)
ax.bar(range(24), internal / 1e3, color='#0072b2', width=0.8, label=f'internal ({internal.sum():,} cars)')
ax.bar(range(24), exchange / 1e3, bottom=internal / 1e3, color='#c27c00', width=0.8, label=f'exchange ({exchange.sum():,} cars)')
ax.set(xlabel='departure hour', ylabel='cars (thousands)', xticks=range(0, 24, 3))
ax.grid(axis='x', visible=False); ax.legend(loc='upper left')
save(fig, 'fig6_departures'); plt.show()

## Notes and limits
- Validation is against 16 IMD sections only: annual all-vehicle averages, one simulated day, one seed. Hourly profiles and travel times are not validated.
- Open points:
  - Ma-20 E (150361) is at about 0.69. The pre-registered exchange test did not explain the gap (UPDATE 72).
  - Ma-1 urban is at about 0.39.
- The baseline ran on the unclipped v4 OD. 436 inbound exchange cars depart after midnight; this is negligible (UPDATE 74).